<div align="center">

# <span style="color: #3498db;">CA2 - MiniMax (Game) Algorithm</span>

**<span style="color:rgb(247, 169, 0);">[Student Name]</span> - <span style="color:rgb(143, 95, 195);">[Student Number]</span>**

</div>

# <span style="color: #3498db;">Checkers with Minimax Algorithm</span>

In this assignment, your tasks are as follows:

- Work with the **pre-built Checkers game engine** (board, pieces, valid-move generation).
- Implement the **Minimax search algorithm**.
- Extend Minimax with **Alpha–Beta pruning**.
- And also, experiment with the **evaluation function** to fine tune the AI's playing style to make it as good as possible. (different heuristics)

In [13]:
import time
import pygame
import matplotlib.pyplot as plt

from copy import deepcopy

RED = (255,0,0)
WHITE = (255, 255, 255)

FPS = 60

# <span style="color: #126ca8; font-size:32px;">Actual Game Implementation</span>

In [14]:

WIDTH, HEIGHT = 800, 800
ROWS, COLS = 8, 8
SQUARE_SIZE = WIDTH//COLS

# rgb
RED = (255, 0, 0)
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
BLUE = (0, 0, 255)
GREY = (128,128,128)

CROWN = pygame.transform.scale(pygame.image.load('assets/crown.png'), (44, 25))

FPS = 60

WIN = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption('Checkers')

In [15]:
def get_row_col_from_mouse(pos):
    x, y = pos
    row = y // SQUARE_SIZE
    col = x // SQUARE_SIZE
    return row, col

In [16]:
class Piece:
    PADDING = 15
    OUTLINE = 2

    def __init__(self, row, col, color):
        self.row = row
        self.col = col
        self.color = color
        self.king = False
        self.x = 0
        self.y = 0
        self.calc_pos()

    def calc_pos(self):
        self.x = SQUARE_SIZE * self.col + SQUARE_SIZE // 2
        self.y = SQUARE_SIZE * self.row + SQUARE_SIZE // 2

    def make_king(self):
        self.king = True

    def draw(self, win):
        radius = SQUARE_SIZE//2 - self.PADDING
        pygame.draw.circle(win, GREY, (self.x, self.y), radius + self.OUTLINE)
        pygame.draw.circle(win, self.color, (self.x, self.y), radius)
        if self.king:
            win.blit(CROWN, (self.x - CROWN.get_width()//2, self.y - CROWN.get_height()//2))

    def move(self, row, col):
        self.row = row
        self.col = col
        self.calc_pos()

    def __repr__(self):
        return str(self.color)

In [17]:
class Board:
    def __init__(self):
        self.board = []
        self.red_left = self.white_left = 12
        self.red_kings = self.white_kings = 0
        self.create_board()

    def draw_squares(self, win):
        win.fill(BLACK)
        for row in range(ROWS):
            for col in range(row % 2, COLS, 2):
                pygame.draw.rect(win, RED, (row*SQUARE_SIZE, col *SQUARE_SIZE, SQUARE_SIZE, SQUARE_SIZE))

    def evaluate(self):
        """
        Evaluation from WHITE's perspective:
        > 0  => White is better
        < 0  => Red is better
        ~ 0  => Equal
        Students must experiment with different hueristics to improve the evaluation function.
        """
        """
        Evaluation from WHITE's perspective:
        > 0  => White is better
        < 0  => Red is better
        ~ 0  => Equal
        """
        # Heuristic weights
        PIECE_VALUE = 10
        KING_VALUE = 15
        EDGE_BONUS = 2      # Pieces on edges are safer
        BACK_ROW_BONUS = 3  # Pieces close to becoming king
    
        white_score = 0
        red_score = 0
    
        for row in range(ROWS):
            for col in range(COLS):
                piece = self.board[row][col]
                if piece != 0:
                    # Base piece value
                    if piece.king:
                        value = KING_VALUE
                    else:
                        value = PIECE_VALUE
                    
                    # Positional bonus
                    if piece.color == WHITE:
                        white_score += value
                        # Bonus for being close to opponent's back row (row 7 for white)
                        if not piece.king:
                            white_score += (row * BACK_ROW_BONUS) / (ROWS - 1)
                        # Edge bonus
                        if col == 0 or col == COLS - 1:
                            white_score += EDGE_BONUS
                    else:  # RED
                        red_score += value
                        # Bonus for being close to opponent's back row (row 0 for red)
                        if not piece.king:
                            red_score += ((ROWS - 1 - row) * BACK_ROW_BONUS) / (ROWS - 1)
                        # Edge bonus
                        if col == 0 or col == COLS - 1:
                            red_score += EDGE_BONUS
        
        # Return score from WHITE's perspective
        return white_score - red_score

    def get_all_pieces(self, color):
        pieces = []
        for row in self.board:
            for piece in row:
                if piece != 0 and piece.color == color:
                    pieces.append(piece)
        return pieces

    def move(self, piece, row, col):
        self.board[piece.row][piece.col], self.board[row][col] = self.board[row][col], self.board[piece.row][piece.col]
        piece.move(row, col)

        if row == ROWS - 1 or row == 0:
            piece.make_king()
            if piece.color == WHITE:
                self.white_kings += 1
            else:
                self.red_kings += 1 

    def get_piece(self, row, col):
        return self.board[row][col]

    def create_board(self):
        for row in range(ROWS):
            self.board.append([])
            for col in range(COLS):
                if col % 2 == ((row +  1) % 2):
                    if row < 3:
                        self.board[row].append(Piece(row, col, WHITE))
                    elif row > 4:
                        self.board[row].append(Piece(row, col, RED))
                    else:
                        self.board[row].append(0)
                else:
                    self.board[row].append(0)

    def draw(self, win):
        self.draw_squares(win)
        for row in range(ROWS):
            for col in range(COLS):
                piece = self.board[row][col]
                if piece != 0:
                    piece.draw(win)

    def remove(self, pieces):
        for piece in pieces:
            if piece:
                self.board[piece.row][piece.col] = 0

                if piece.color == RED:
                    self.red_left -= 1
                    if piece.king:
                        self.red_kings -= 1
                else:
                    self.white_left -= 1
                    if piece.king:
                        self.white_kings -= 1

    def winner(self):
        if self.red_left <= 0:
            return WHITE
        if self.white_left <= 0:
            return RED
        return None

    def get_valid_moves(self, piece):
        """
        Get all valid moves for a given piece as:
        { (row, col): [list of captured pieces] }.
        """
        moves = {}
        left = piece.col - 1
        right = piece.col + 1
        row = piece.row

        if piece.color == RED or piece.king:
            moves.update(self._traverse_left(row -1, max(row-3, -1), -1, piece.color, left))
            moves.update(self._traverse_right(row -1, max(row-3, -1), -1, piece.color, right))
        if piece.color == WHITE or piece.king:
            moves.update(self._traverse_left(row +1, min(row+3, ROWS), 1, piece.color, left))
            moves.update(self._traverse_right(row +1, min(row+3, ROWS), 1, piece.color, right))

        return moves

    def _traverse_left(self, start, stop, step, color, left, skipped=[]):
        moves = {}
        last = []
        for r in range(start, stop, step):
            if left < 0:
                break

            current = self.board[r][left]
            if current == 0:
                if skipped and not last:
                    break
                elif skipped:
                    moves[(r, left)] = last + skipped
                else:
                    moves[(r, left)] = last

                if last:
                    if step == -1:
                        row = max(r-3, 0)
                    else:
                        row = min(r+3, ROWS)
                    moves.update(self._traverse_left(r+step, row, step, color, left-1,skipped=last))
                    moves.update(self._traverse_right(r+step, row, step, color, left+1,skipped=last))
                break
            elif current.color == color:
                break
            else:
                last = [current]

            left -= 1

        return moves

    def _traverse_right(self, start, stop, step, color, right, skipped=[]):
        moves = {}
        last = []
        for r in range(start, stop, step):
            if right >= COLS:
                break

            current = self.board[r][right]
            if current == 0:
                if skipped and not last:
                    break
                elif skipped:
                    moves[(r,right)] = last + skipped
                else:
                    moves[(r, right)] = last

                if last:
                    if step == -1:
                        row = max(r-3, 0)
                    else:
                        row = min(r+3, ROWS)
                    moves.update(self._traverse_left(r+step, row, step, color, right-1,skipped=last))
                    moves.update(self._traverse_right(r+step, row, step, color, right+1,skipped=last))
                break
            elif current.color == color:
                break
            else:
                last = [current]

            right += 1

        return moves

In [18]:
class Game:
    def __init__(self, win):
        self._init()
        self.win = win

    def update(self):
        self.board.draw(self.win)
        self.draw_valid_moves(self.valid_moves)
        pygame.display.update()

    def _init(self):
        self.selected = None
        self.board = Board()
        self.turn = RED
        self.valid_moves = {}

    def winner(self):
        piece_winner = self.board.winner()
        if piece_winner is not None:
            return piece_winner

        moves = 0
        for p in self.board.get_all_pieces(self.turn):
            moves += len(self.board.get_valid_moves(p))

        if moves == 0:
            return WHITE if self.turn == RED else RED

        return None

    def reset(self):
        self._init()

    def select(self, row, col):
        if self.selected:
            result = self._move(row, col)
            if not result:
                self.selected = None
                self.select(row, col)

        piece = self.board.get_piece(row, col)
        if piece != 0 and piece.color == self.turn:
            self.selected = piece
            self.valid_moves = self.board.get_valid_moves(piece)
            return True

        return False

    def _move(self, row, col):
        piece = self.board.get_piece(row, col)
        if self.selected and piece == 0 and (row, col) in self.valid_moves:
            self.board.move(self.selected, row, col)
            skipped = self.valid_moves[(row, col)]
            if skipped:
                self.board.remove(skipped)
            self.change_turn()
        else:
            return False

        return True

    def draw_valid_moves(self, moves):
        for move in moves:
            row, col = move
            pygame.draw.circle(self.win, BLUE, (col * SQUARE_SIZE + SQUARE_SIZE//2, row * SQUARE_SIZE + SQUARE_SIZE//2), 15)

    def change_turn(self):
        self.valid_moves = {}
        if self.turn == RED:
            self.turn = WHITE
        else:
            self.turn = RED

    def get_board(self):
        return self.board

    def ai_move(self, board):
        self.board = board
        self.change_turn()

# <span style="color: #126ca8; font-size:32px;">Minimax AI Implementation</span>

In [19]:
# YOU MUST COMPLETE THE MINIMAX SECTION FULLY AND ALSO MAKE SURE TO IMPLEMENT ALPHA BETA PRUNING IN THE OPTIMIZED SECTION

# GIVEN HELPER FUNCTIONS FOR YOUR TO USE IN YOUR MINIMAX IMPLIMENTATION

def simulate_move(piece, move, board, skip):
    """
    Simulate moving 'piece' to 'move' on a cloned board, including captures.
    This is used by the search to explore future positions.
    """
    board.move(piece, move[0], move[1])
    if skip:
        board.remove(skip)

    return board


def get_all_moves(board, color):
    """
    Generate all possible next board positions for 'color'.
    """
    moves = []

    for piece in board.get_all_pieces(color):
        valid_moves = board.get_valid_moves(piece)
        for move, skip in valid_moves.items():
            temp_board = deepcopy(board)
            temp_piece = temp_board.get_piece(piece.row, piece.col)
            new_board = simulate_move(temp_piece, move, temp_board, skip)
            moves.append(new_board)

    return moves

# <span style="color: #126ca8; font-size:32px;">Base (No Alpha/Beta Pruning)</span>

In [20]:
# ================================
# ================================
# Requirements:
# - max_player: True => WHITE is the maximizing player
# - max_player: False => RED is the minimizing player
# - Use Board.evaluate() as the heuristic evaluation function.
# - Use get_all_moves(board, color) to get next positions.

def minimax(position, depth, max_player: bool):
     # Base case: depth is 0 or game is over
    if depth == 0 or position.winner() is not None:
        return position.evaluate(), None
    
    if max_player:
        max_eval = float('-inf')
        best_move = None
        for new_board in get_all_moves(position, WHITE):
            eval, _ = minimax(new_board, depth - 1, False)
            if eval > max_eval:
                max_eval = eval
                best_move = new_board
        return max_eval, best_move
    else:
        min_eval = float('inf')
        best_move = None
        for new_board in get_all_moves(position, RED):
            eval, _ = minimax(new_board, depth - 1, True)
            if eval < min_eval:
                min_eval = eval
                best_move = new_board
        return min_eval, best_move

    

# <span style="color: #126ca8; font-size:32px;">Optimized Version (With Alpha/Beta Pruning)</span>

In [21]:
# ======================================
# ======================================
# Implement minimax_alpha_beta with:
# - Same behavior as minimax, but with pruning using (alpha, beta).
# - alpha: best value that the maximizer currently can guarantee.
# - beta : best value that the minimizer currently can guarantee.
# - Prune (break early) as seen in your course materials.


def minimax_alpha_beta(position, depth, alpha, beta, max_player: bool):
     # Base case: depth is 0 or game is over
    if depth == 0 or position.winner() is not None:
        return position.evaluate(), None
    
    if max_player:
        max_eval = float('-inf')
        best_move = None
        for new_board in get_all_moves(position, WHITE):
            eval, _ = minimax_alpha_beta(new_board, depth - 1, alpha, beta, False)
            if eval > max_eval:
                max_eval = eval
                best_move = new_board
            alpha = max(alpha, eval)
            if beta <= alpha:
                break  # Prune remaining branches
        return max_eval, best_move
    else:
        min_eval = float('inf')
        best_move = None
        for new_board in get_all_moves(position, RED):
            eval, _ = minimax_alpha_beta(new_board, depth - 1, alpha, beta, True)
            if eval < min_eval:
                min_eval = eval
                best_move = new_board
            beta = min(beta, eval)
            if beta <= alpha:
                break  # Prune remaining branches
        return min_eval, best_move
     

In [22]:
# In this section you should check how your minimax with and without alpha-beta pruning function works.
# You can use the provided checkers game function below also you can also do plots to better show the results.

def play_ai_vs_ai(search_depth=4, use_pruning=True, track_nodes=False):
    """
    Plays a full Checkers game: WHITE AI vs RED AI (both use minimax_alpha_beta also try without alpha-beta pruning).
    Returns:
        winner_color  -> WHITE, RED, or None (for draw/stuck)
        moves_count   -> number of half-moves played
        elapsed_time  -> total game time in seconds
    """
    board = Board()
    turn = RED  # same as in Game._init
    moves_count = 0

    # Optional: node counter good to have :)
    if track_nodes:
        global NODES_SEEN
        NODES_SEEN = 0

    start_time = time.time()

    while True:
        winner = board.winner()
        if winner is not None:
            elapsed = time.time() - start_time
            return winner, moves_count, elapsed

        all_moves = get_all_moves(board, turn)
        if not all_moves:
            elapsed = time.time() - start_time
            winner = WHITE if turn == RED else RED
            return winner, moves_count, elapsed

        if use_pruning:
            _, best_board = minimax_alpha_beta(
                position=board,
                depth=search_depth,
                alpha=float('-inf'),
                beta=float('inf'),
                max_player=(turn == WHITE)
            )
        else:
            _, best_board = minimax(
                position=board,
                depth=search_depth,
                max_player=(turn == WHITE)
            )

        if best_board is None:
            best_board = all_moves[0]

        board = best_board
        moves_count += 1

        turn = WHITE if turn == RED else RED

# Use the above to complete these to show that your AI can play the game correctly and efficiently.
# You should be able to see that your AI makes good moves.

def print_checkers_results(results, depth, pruning, total_time, avg_moves):
    print("=" * 60)
    print(f"Results for Depth={depth}, Pruning={'ON' if pruning else 'OFF'}")
    print("=" * 60)
    print(f"Total games played: {len(results)}")
    print(f"White wins: {results.count(WHITE)}")
    print(f"Red wins:  {results.count(RED)}")
    print(f"Draws:     {results.count(None)}")
    print(f"Total time: {total_time:.4f} seconds")
    print(f"Average moves per game: {avg_moves:.2f}")
    print(f"Average time per move: {total_time / max(1, sum(results != None for results in results)):.4f} sec")
    print("=" * 60)

def check_checkers_results(max_depth=3, games_per_setting=5):
    print("\n" + "=" * 80)
    print("CHECKERS AI PERFORMANCE ANALYSIS")
    print("=" * 80)
    
    # Test without pruning
    print("\n--- WITHOUT Alpha-Beta Pruning ---")
    for depth in range(1, max_depth + 1):
        results = []
        total_time = 0
        total_moves = 0
        
        for game_num in range(games_per_setting):
            winner, moves, elapsed = play_ai_vs_ai(
                search_depth=depth,
                use_pruning=False,
                track_nodes=False
            )
            results.append(winner)
            total_time += elapsed
            total_moves += moves
        
        avg_moves = total_moves / games_per_setting
        print_checkers_results(results, depth, False, total_time, avg_moves)
    
    # Test with pruning
    print("\n--- WITH Alpha-Beta Pruning ---")
    for depth in range(1, max_depth + 1):
        results = []
        total_time = 0
        total_moves = 0
        
        for game_num in range(games_per_setting):
            winner, moves, elapsed = play_ai_vs_ai(
                search_depth=depth,
                use_pruning=True,
                track_nodes=False
            )
            results.append(winner)
            total_time += elapsed
            total_moves += moves
        
        avg_moves = total_moves / games_per_setting
        print_checkers_results(results, depth, True, total_time, avg_moves)

def run_analysis_for_plots(max_depth=4, games_per_setting=5):
    """
     Collects data for plotting graphs showing:
     - Depth vs Game Time (with/without pruning)
     - Depth vs Win Rate for White
     """
    depths = list(range(1, max_depth + 1))
    
    times_no_pruning = []
    times_with_pruning = []
    win_rates_no_pruning = []
    win_rates_with_pruning = []
    
    for depth in depths:
        # Without pruning
        results_no = []
        time_no = 0
        for _ in range(games_per_setting):
            winner, _, elapsed = play_ai_vs_ai(
                search_depth=depth,
                use_pruning=False,
                track_nodes=False
            )
            results_no.append(winner)
            time_no += elapsed
        times_no_pruning.append(time_no / games_per_setting)
        win_rates_no_pruning.append(results_no.count(WHITE) / games_per_setting * 100)
        
        # With pruning
        results_pr = []
        time_pr = 0
        for _ in range(games_per_setting):
            winner, _, elapsed = play_ai_vs_ai(
                search_depth=depth,
                use_pruning=True,
                track_nodes=False
            )
            results_pr.append(winner)
            time_pr += elapsed
        times_with_pruning.append(time_pr / games_per_setting)
        win_rates_with_pruning.append(results_pr.count(WHITE) / games_per_setting * 100)
    
    # Plot results
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Time comparison
    ax1.plot(depths, times_no_pruning, 'r-o', label='Without Pruning')
    ax1.plot(depths, times_with_pruning, 'b-o', label='With Alpha-Beta')
    ax1.set_xlabel('Search Depth')
    ax1.set_ylabel('Average Game Time (seconds)')
    ax1.set_title('Game Time vs Search Depth')
    ax1.legend()
    ax1.grid(True)
    
    # Plot 2: Win rate comparison
    ax2.plot(depths, win_rates_no_pruning, 'r-o', label='Without Pruning')
    ax2.plot(depths, win_rates_with_pruning, 'b-o', label='With Alpha-Beta')
    ax2.set_xlabel('Search Depth')
    ax2.set_ylabel('White Win Rate (%)')
    ax2.set_title('AI Performance vs Search Depth')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print("\n" + "=" * 70)
    print("ANALYSIS SUMMARY")
    print("=" * 70)
    print(f"At depth {max_depth}, pruning reduces game time by {((times_no_pruning[-1] - times_with_pruning[-1]) / times_no_pruning[-1] * 100):.1f}%")
    print(f"Win rate with pruning at depth {max_depth}: {win_rates_with_pruning[-1]:.1f}%")
    print("=" * 70)
    
    return {
        'depths': depths,
        'times_no_pruning': times_no_pruning,
        'times_with_pruning': times_with_pruning,
        'win_rates_no_pruning': win_rates_no_pruning,
        'win_rates_with_pruning': win_rates_with_pruning
    }

# <span style="color: #126ca8; font-size:32px;">Main Game Loop</span>

In [23]:
run = True
clock = pygame.time.Clock()
game = Game(WIN)

SEARCH_DEPTH = 4  # Experiment with this depth to see how it affects the game's performance and choose what you deem best.

# ALSO MAKE SURE TO HAVE A MAIN VERSION WITH ALPHA BETA PRUNING SO WE CAN GRADE YOU BASED ON IT!

while run:
    clock.tick(FPS)

    if game.turn == WHITE:
        # AI plays as WHITE using alpha-beta pruning
        _, new_board = minimax_alpha_beta(
            game.get_board(),
            depth=SEARCH_DEPTH,
            alpha=float('-inf'),
            beta=float('inf'),
            max_player=True
        )
        game.ai_move(new_board)

    if game.winner() is not None:
        print("Winner:", "WHITE" if game.winner() == WHITE else "RED")
        run = False

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            run = False

        if event.type == pygame.MOUSEBUTTONDOWN:
            pos = pygame.mouse.get_pos()
            row, col = get_row_col_from_mouse(pos)
            game.select(row, col)

    game.update()

pygame.quit()